# Debug 469

In [1]:
import os
from list_active_ifps import list_active_ifps
from save_ifps_to_disk import save_ifps_to_disk
from forecast_ifp import forecast_ifp
from gather_news_for_ifps import gather_news_for_ifps

loading massive wiki index 2025-07-17 21:51:26.166135
loading wiki article titles 2025-07-17 21:52:03.855786
loading sentence transformer model 2025-07-17 21:52:07.157335
done 2025-07-17 21:52:08.656716


In [2]:
    os.makedirs('glimt/prompt', exist_ok=True)
    ifps = list_active_ifps()
    id_to_ifp = save_ifps_to_disk(ifps)
    news = gather_news_for_ifps(ifps)
    for ifp in ifps:
        if ifp['id'] == 469:
            break

200


In [3]:
ifp

{'id': 469,
 'type': 'bins',
 'symbol': 'War_KURSK_Jan_25',
 'state': 'active',
 'dates': {'startDay': 20250114,
  'endDay': 20251230,
  'scoringStartDay': 20250114,
  'scoringEndDay': 20251230},
 'props': {'title': 'By what date will Russia regain control of the Kursk region?',
  'ai_title': '',
  'shortTitle': 'When will Russia push Ukraine out of Kursk?',
  'details': '<b>Resolution:</b>\nThe question will resolve in relation to the date when full Russian control is established of all parts of Kursk region, as reported by official sources and confirmed by <a target=_new href="https://liveuamap.com/">Livemap</a>.\n\n<b>Background:</b> On August 6, 2024, the Ukrainian Armed Forces crossed the Russian-Ukrainian border near the city of Sudzha. Ukrainian troops began advancing deep into Russian territory and within a few days controlled several hundred square kilometers. This is the first Ukrainian combined-arms operation inside Russian territorysince the beginning of the full-scale Russ

In [4]:
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from humor_me import humor_me
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from jsx_request import jsx_request
from jsx_forecast import jsx_forecast
from datetime import datetime

In [ ]:
    print('begin FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())
    title_plus_criteria = detailed_proposition(ifp)
    wiki_articles = wiki_semantic_search(title_plus_criteria)
    ifp_news_sources, ifp_news_text = split_news_into_text_and_urls(ifp, news)
    source_summaries, sources = create_source_summaries(ifp['id'], title_plus_criteria, wiki_articles, ifp_news_sources, ifp_news_text)
    rephrase_binary_outcomes(ifp)
    research = format_research(source_summaries)

In [7]:
import json
from saved_prompt import saved_prompt
from get_periods_of_whenwill_question import get_periods_of_whenwill_question
from filter_periods_to_today import filter_periods_to_today
from get_event import get_event

In [9]:
    (fn, savep) = saved_prompt(ifp)

In [10]:
savep

['\nYou are a talented, experienced and confident superforecaster. You are asked a question:\n\n```question\nBy what date will Russia regain control of the Kursk region?\n```\n\nYou are given details on how to interpret the terms of the question:\n\n```details\n<b>Resolution:</b>\nThe question will resolve in relation to the date when full Russian control is established of all parts of Kursk region, as reported by official sources and confirmed by <a target=_new href="https://liveuamap.com/">Livemap</a>.\n\n<b>Background:</b> On August 6, 2024, the Ukrainian Armed Forces crossed the Russian-Ukrainian border near the city of Sudzha. Ukrainian troops began advancing deep into Russian territory and within a few days controlled several hundred square kilometers. This is the first Ukrainian combined-arms operation inside Russian territorysince the beginning of the full-scale Russian invasion of Ukraine. Putin has repeatedly publicly claimed to have regained control, but by the end of 2024, 

In [ ]:
    if savep: return savep
    ## Prompt with rationale fields split into for and against
    details = ifp['props']['details']
    title = ifp['props']['title']
    bins = [x['props']['title'] for x in ifp['bins']]
    rejected = []
    if 'When' or "By what date" in title:
        print('howdy!')

SyntaxError: 'return' outside function (2635584789.py, line 2)

In [ ]:
        periods = get_periods_of_whenwill_question(ifp)
        rejected, filtered = filter_periods_to_today(periods)
        original_bins = bins
        event = get_event(ifp)
        bins = [f"{event} between {b} and {c}" for a,b,c in filtered]
    sb1 = '\n'.join([f"""* O{i+1}. {bin}""" for i, bin in enumerate(bins)])
    psum = '+'.join([f'P{i+1}' for i, bin in enumerate(bins)])
    pcom = ','.join([f'P{i+1}' for i, bin in enumerate(bins)])
    sbins = f"""The question has one of {len(bins)} outcomes namely  

{sb1}

Each outcome Oi has a probability Pi where 0 <= Pi <= 1.
We must have that {psum} = 1.0.
Add some reasonable amount of randomness/noise to the estimation of the branches.
The output is a Python list wrapped by a binProbs tag, in this format:
```binProbs
[{pcom}]
```
"""
    prompt = f"""
You are a talented, experienced and confident superforecaster. You are asked a question:

```question
{title}
```

You are given details on how to interpret the terms of the question:

```details
{details}
```

Your assistant has research related news and Wikipedia articles and prepared summaries of each one.
Use the data in these research summaries to analyse the question:

{research}

For you to be marked Successful, you must output 3 things:

1. Probabilities for the outcomes of the question.  
{sbins}

2. Reasons your probabilities might be right, wrapped in tag in this format:
```rRight
...reasons you might be right
```

3. Reasons your probabilities might be wrong, wrapped in tag in this format:
```rWrong
...reasons you might be wrong
```
"""
    with open(fn, 'w') as f:
        json.dump((prompt, rejected), f)

In [6]:
rejected

[]

In [ ]:
    ## Run the prompt 5 times
    prompt_tries = 2 # Waste of time on Mistral 4 bit
    answers = [humor_me(prompt, i+1) for i in range(prompt_tries)]
    binProbs = [get_bin_probs(a) for a in answers]
    rights = [get_rights(a) for a in answers]
    wrongs = [get_wrongs(a) for a in answers]
    
    ## Median forecasts and rationales
    forecast = rejected + median_forecast(binProbs)
    
    right = median_rationale(rights)
    wrong = median_rationale(wrongs)
    jsx_request(jsx_forecast(ifp['id'],forecast,right,wrong,sources))
    print('end FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())

    result = (forecast, right, wrong, sources)
    fn = f'glimt/forecast'
    import os
    os.makedirs(fn, exist_ok=True)
    fn = f"{fn}/{ifp['id']}.json"

    import json
    with open(fn, 'w') as f:
        json.dump(result, f)

    return result